# Burgers: how many output modes are needed?

We learn the solution operator $K: u_0 \mapsto u(\cdot, T)$ of the viscous Burgers' equation
$u_t + u u_x = \nu u_{xx}$ on $(0,1)$, $u(0,t) = u(1,t) = 0$, with $T = 0.2$.

* Inputs and outputs are sine coefficients in the basis $\sqrt 2 \sin(n\pi x)$; data come from the spectral Galerkin solver in `burgers_solver.py`.
* Input measure: $d_{\mathrm{in}} = 20$ independent coefficients, the $n$-th distributed as $\mathrm{Jacobi}(\alpha_n, \alpha_n)$. **As run:** $\alpha_n = (n-1)^2$, $n = 1, \dots, 20$ (so the first mode is uniform on $[-1,1]$).
* Approximation space: polynomial operators $\psi_m\, p_\lambda(u_0)$ with $\lambda$ in an anisotropic index set with weights $w_j$ = `linspace(1, 0.01, 20)` (the weights enter as an exponent $1/w_j$, see `owls.index_set`) and univariate degree at most 10.

For $\nu \in \{1, 0.1, 0.01, 0.001\}$ we estimate the fraction of the output energy lost when the output is truncated to its first $d_{\mathrm{out}}$ sine modes,
$$1 - \frac{\sum_{j \le d_{\mathrm{out}}} \mathbb E\, |\langle \psi_j, K(u_0)\rangle|^2}{\mathbb E\, \|K(u_0)\|^2}, \qquad u_0 \sim \rho,$$
with the expectations replaced by averages over 100 Monte Carlo samples (500 output modes are computed).

**Status:** rerun after the cleanup. The arXiv v1 version averaged over samples from the optimal measure $\mu$ instead of $\rho$. See `ERRATA.md`.

In [ ]:
RECOMPUTE = False  # True: rerun the experiment and overwrite data/ (see ../README.md for run times)
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig
from burgers_solver import burgers_map, sine_basis_vandermonde

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters (as run, apart from sampling from rho)
NUS = [1, 0.1, 0.01, 0.001]
D_OUT = 500           # output modes kept
D_SOLVE = 500         # as run: max(10 * D_IN, D_OUT)
T, DT = 0.2, 1e-4
M = 100               # Monte Carlo samples u_0 ~ rho
THRESHOLD = 0.999     # the horizontal line marks 0.1% energy lost
SEED = 0
if QUICK:
    NUS, D_OUT, D_SOLVE, T, M = [0.1, 0.01], 100, 100, 0.01, 10

D_IN = 20                                             # input sine modes
MAX_DEGREE = 10                                       # max univariate polynomial degree
measures = [owls.Jacobi(a**2) for a in range(D_IN)]  # as run: alpha_n = (n - 1)^2 for n = 1..20
weights = np.linspace(1, 0.01, D_IN)                 # anisotropy weights

# Input truncation. The paper's runs use D_IN = 20 input modes. Under alpha_n = (n-1)^2 (as run),
# these carry the following fraction of the input energy sum_n 1/(2 alpha_n + 3):
n = np.arange(1, 2_000_001)
var = 1 / (2.0 * (n - 1) ** 2 + 3)
print(f"{D_IN} input modes carry {var[:D_IN].sum() / var.sum():.1%} of the input energy")

In [ ]:
if RECOMPUTE or QUICK:
    energy = np.zeros((len(NUS), D_OUT))
    for i, nu in enumerate(NUS):
        u0 = owls.sample_rho(M, measures, np.random.default_rng([SEED, i]))
        energy[i] = np.mean(burgers_map(u0, nu, T, DT, D_SOLVE, D_OUT) ** 2, axis=0)  # E |<psi_j, K(u_0)>|^2
    np.save(os.path.join(DATA, "output_energy.npy"), energy)

In [ ]:
energy = np.load(os.path.join(DATA, "output_energy.npy"))
fraction_lost = 1 - np.cumsum(energy, axis=1) / np.sum(energy, axis=1, keepdims=True)
d_out = np.arange(1, energy.shape[1] + 1)

fig_width, fig_height = paper_style(width=0.5, aspect=0.6, fontsize=7, legend_fontsize=5)
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout="constrained")
colors, markers = ["darkred", "firebrick", "indianred", "lightcoral"], ["D", "^", "o", "*"]
thresh_idxs = []
for i, nu in enumerate(NUS):
    ax.plot(d_out, fraction_lost[i], color=colors[i], marker=markers[i], linestyle="--", markersize=3, linewidth=1, label=rf"$\nu$ = {nu}")
    thresh_idxs.append(np.argmax(fraction_lost[i] < 1 - THRESHOLD))
    print(f"nu = {nu}: d_out = {d_out[thresh_idxs[-1]]} loses less than {100 * (1 - THRESHOLD):.1f}% of the output energy")
ax.hlines([1 - THRESHOLD], xmin=0, xmax=d_out[-1], color="k", label=r"$0.1\%$")
ax.set(yscale="log", ylim=(1e-8, 1), xlim=(-1, 1.25 * d_out[max(thresh_idxs)]), xlabel=r"$d_{\mathrm{out}}$", ylabel=r"Energy Fraction Lost")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="best")
savefig(os.path.join(FIGS, "vb_out_truc_err.pdf"))
plt.show()